## Imports

In [2]:
import os
import sys
sys.path.insert(0, os.path.abspath('/home/ec2-user/multimodal-rag'))

import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import json
from tqdm.notebook import tqdm
from scripts.vlm import VLM

## Configurations

In [3]:
SEED = 19
QUESTION_PATHS = {
    "FinancialReport" : "../data/raw/REAL-MM-RAG_FinReport/test/qas.jsonl",
    "TechReport" : "../data/raw/REAL-MM-RAG_TechReport/test/qas.jsonl",
    "TechSlides" : "../data/raw/REAL-MM-RAG_TechSlides/test/qas.jsonl"
}
RESULTS_PATH = "./results/baseline_no_rag"
model_id = "Qwen/Qwen3-VL-8B-Instruct"
checkpoint_path = "../Qwen3-VL-8B-Instruct"

## Load Data

In [4]:
dataframes = {}

for key, path in QUESTION_PATHS.items():
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            records.append({
                "example_index": obj.get("example_index"),
                "question": obj.get("question"),
                "answer": obj.get("answer"),
                "intended_img": obj.get("image_path")
            })
    dataframes[key] = pd.DataFrame(records).set_index("example_index")

Split into test and train

In [5]:
splits = {}
for key, df in dataframes.items():
    train = df.sample(frac=0.8, random_state=SEED)
    test = df.drop(train.index)
    splits[key] = {"train": train, "test": test}

## Inference

In [6]:
vlm = VLM(model_id, checkpoint_path)

--- Local model found at: ../Qwen3-VL-8B-Instruct ---
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Multi-thread loading shards: 100% Completed | 4/4 [02:10<00:00, 32.58s/it]
Capturing batches (bs=24 avail_mem=1.75 GB):   0%|          | 0/7 [00:00<?, ?it/s]2026-04-12 21:35:12,293 - CUTE_DSL - WARNING - [handle_import_error] - Unexpected error during package walk: cutlass.cute.experimental
[2026-04-12 21:35:12] Unexpected error during package walk: cutlass.cute.experimental
Capturing batches (bs=1 avail_mem=1.68 GB): 100%|██████████| 7/7 [00:05<00:00,  1.39it/s] 


In [ ]:
os.makedirs(RESULTS_PATH, exist_ok=True)

all_results = []

for key, split in splits.items():
    test_df = split['test']
    print(f"Running Inference for {key}")
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": row['question']}
                ]
            }
        ]
        response = vlm.generate(messages)
        all_results.append({
            "dataset": key,
            "question": row['question'],
            "generated_answer": response['text'],
            "ground_truth": row['answer']
        })

df_results = pd.DataFrame(all_results)
df_results.to_csv(f"{RESULTS_PATH}/baseline_no_rag_results.csv", index=False)

Running Inference for FinancialReport


  0%|          | 0/171 [00:00<?, ?it/s]

Running Inference for TechReport


  0%|          | 0/259 [00:00<?, ?it/s]

Running Inference for TechSlides


  0%|          | 0/271 [00:00<?, ?it/s]

## Evaluation